# Comparison with Observations: Dynamics (only)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

import utilities.plot_settings

## Loading the observed data

Compare the spatial and proper motion distributions of simulated pulsars with neutron stars in the ATNF catalogue. We select from the ATNF Catalogue all neutron stars that are presumably isolated and not recycled. We also compare with the subsample of those that have a measured proper motion.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)

df_atnf.head()
df_atnf.columns = df_atnf.columns.droplevel(1)

In [ ]:
# Select only stars with measured P, Pdot, DM and radio flux.
# We also select only those that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["P0"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["P1"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["DIST"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["ASSOC"].str.match("|".join(discard))]

RA_atnf = df_atnf["RAJD"].to_numpy().astype(np.float64)
DEC_atnf = df_atnf["DECJD"].to_numpy().astype(np.float64)
P_atnf = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_atnf = df_atnf["P1"].to_numpy().astype(np.float64)
dist_atnf = df_atnf["DIST"].to_numpy().astype(np.float64)

# Select only isolated non-recycled neutron stars, i.e., those with Pdot > 1e-17.
# We also select only those with a distance estimate (deduced from the DM) which is < 25 kpc.
cond = (Pdot_atnf > 1e-17) & (dist_atnf < 25)
RA_atnf = RA_atnf[cond]
DEC_atnf = DEC_atnf[cond]
dist_atnf = dist_atnf[cond]

In [ ]:
# Read observed proper motion .csv file.
data_pm = pd.read_csv(
    "../../data/observations/PSRs_prop_motion_22-05-2020.csv", header=[0, 1]
)
data_pm.head()

# Select only stars with measured P and Pdot and those that are not in globular clusters.
data_pm = data_pm[~data_pm["P0"]["[s]"].isin(["NAN"])]
data_pm = data_pm[~data_pm["P1"]["[s/s]"].isin(["NAN"])]
data_pm = data_pm[~data_pm["DIST_DM"]["[kpc]"].isin(["NAN"])]
data_pm = data_pm[
    ~data_pm["ASSOC"]["Unnamed: 24_level_1"].isin(
        [
            "EXGAL:SMC",
            "EXGAL:LMC",
            "GC:47Tuc",
            "GC:M3",
            "GC:M5",
            "GC:M13",
            "GC:NGC6440",
            "GC:Ter5",
            "GC:NGC6441",
            "GC:NGC6517",
            "GC:NGC6522",
            "GC:NGC6624",
            "GC:M28(NGC6626)",
            "GC:NGC6652",
            "GC:M22(NGC6656)",
            "GC:NGC6752",
            "GC:NGC6760",
            "GC:M15",
            "GC:M30",
        ]
    )
]

# Extract parameters.
RA_pm = data_pm["RAJD"]["[deg]"].to_numpy().astype(np.float64)
DEC_pm = data_pm["DECJD"]["[deg]"].to_numpy().astype(np.float64)
pmRA_pm = data_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float64)
pmRA_err_pm = data_pm["PMRA_err"]["[mas/yr]"].to_numpy().astype(np.float64)
pmDEC_pm = data_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float64)
pmDEC_err_pm = data_pm["PMDEC_err"]["[mas/yr]"].to_numpy().astype(np.float64)
dist_pm = data_pm["DIST_DM"]["[kpc]"].to_numpy().astype(np.float64)
NS_class = data_pm["CLASS"]["Unnamed: 13_level_1"].to_numpy()
P_pm = data_pm["P0"]["[s]"].to_numpy().astype(np.float64)
Pdot_pm = data_pm["P1"]["[s/s]"].to_numpy().astype(np.float64)
assoc = data_pm["ASSOC"]["Unnamed: 24_level_1"].to_numpy()

# Select only isolated non-recycled neutron stars, i.e., those with Pdot > 1e-17.
# We also select only those with a distance estimate (deduced from the DM) which is <25 kpc.
cond = (Pdot_pm > 1e-17) & (NS_class != "Binary PSR") & (dist_pm < 25)
RA_pm = RA_pm[cond]
DEC_pm = DEC_pm[cond]
pmRA_pm = pmRA_pm[cond]
pmRA_err_pm = pmRA_err_pm[cond]
pmDEC_pm = pmDEC_pm[cond]
pmDEC_err_pm = pmDEC_err_pm[cond]
dist_pm = dist_pm[cond]

## Loading the simulation data

In [ ]:
# Select a `final_population.pkl.gz` file for import:
data = pd.read_pickle(
    "../../data/example_simulation_full_edm/final_population.pkl.gz",
    compression="gzip",
)
data.head()

In [ ]:
r = data["r"]["[kpc]"].to_numpy()
phi = data["phi"]["[rad]"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)
z = data["z"]["[kpc]"].to_numpy()
RA = data["ra"]["[deg]"].to_numpy()
DEC = data["dec"]["[deg]"].to_numpy()
pmRA = data["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC = data["pm_dec"]["[mas yr^-1]"].to_numpy()
v_r = data["v_r"]["[km s^-1]"].to_numpy()
v_phi = data["v_phi"]["[km s^-1]"].to_numpy()
v_z = data["v_z"]["[km s^-1]"].to_numpy()
d = data["dist"]["[kpc]"].to_numpy()

Introduce some selection biases as a function of the distance.

In [ ]:
def calculate_selection_weights(d: np.ndarray) -> np.ndarray:
    """
    Calculate the weights assigned to every star for selection.
    Weights are evaluated as a function of distance,
    so that nearest stars are easier and more likely to be detected.

    Args:
        d (np.ndarray): array of distances from the Sun [kpc].

    Returns:
        (np.ndarray): array of selection weights.
    """

    # This function has been fine tuned to match the distance distribution
    # of the neutron stars with observed proper motion.
    weights = np.exp(-0.5 * d) / d

    # Normalize the weights to their sum.
    w = weights / np.sum(weights)

    return w


# Calculate selection weights that are function of the distance.
w = calculate_selection_weights(d)

Select a number of simulated neutron stars equal to the number of observed stars with observer proper motion according to the selection function above. Then compare the distribution of distances of the observed neutron stars and the resampled simulated population with a K-S (Kolmogorov-Smirnov) test. The K-S statistic is averaged over 1000 comparisons.

In [ ]:
# Select a number of simulated NSs equal to those with proper motion according to the selection function.
data_select = data.sample(len(RA_pm), replace=False, weights=w)
d_sel = data_select["dist"]["[kpc]"].to_numpy()

RA_sel = data_select["ra"]["[deg]"].to_numpy()
DEC_sel = data_select["dec"]["[deg]"].to_numpy()
pmRA_sel = data_select["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_sel = data_select["pm_dec"]["[mas yr^-1]"].to_numpy()
vls_sel = data_select["v_ls"]["[km s^-1]"].to_numpy()
d_sel = data_select["dist"]["[kpc]"].to_numpy()

# Compare the distribution of the observed and simulated samples with K-S test.
# We average the statistics over n_trials.
n_trials = 1000
p_values = np.zeros(n_trials)
for i in range(n_trials):
    df_select = data.sample(len(RA_pm), replace=False, weights=w)
    d_sel_test = df_select["dist"]["[kpc]"].to_numpy()
    # Perform the K-S test on the observed and simulated samples.
    stat, p = stats.ks_2samp(dist_pm, d_sel_test)
    p_values[i] = p

print(f"K-S test p-value: {np.mean(p_values)}")

## Visualizing the corresponding populations

### Histogramming the distance from the Sun

We compare the simulated population with all the isolated neutron stars in the ATNF Catalogue (`Observed full ATNF` in the legend) and the ones with measured proper motions (`Observed proper motion`).

In [ ]:
d_edges = np.linspace(0, 25, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_atnf,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed full ATNF",
    rasterized=True,
)
ax.hist(
    dist_pm,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="tab:gray",
    facecolor="tab:gray",
    lw=4,
    alpha=1.0,
    label=r"Observed proper motion",
    rasterized=True,
)
ax.hist(
    d,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated all",
    rasterized=True,
)
ax.hist(
    d_sel,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated + bias",
    zorder=5,
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.set_xlim(0.0, 25.0)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show(block=False)

Plotting the distribution in RA and DEC in ICRS (International Celestial Reference System) coordinates.

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15, 7))

ax.plot(
    RA,
    DEC,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label=r"Simulated all",
)
ax.plot(
    RA_atnf,
    DEC_atnf,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed full ATNF",
)
ax.plot(
    RA_pm,
    DEC_pm,
    linestyle="None",
    marker="x",
    color="black",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed proper motion",
)
ax.plot(
    RA_sel,
    DEC_sel,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Simulated + bias",
)


ax.plot(
    RA_galcen,
    DEC_galcen,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Galactic center",
)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20, markerscale=1
)

plt.show()

Histrogramming RA and DEC coordinate positions.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_atnf,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    RA_pm,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="tab:gray",
    facecolor="tab:gray",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    RA,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    RA_sel,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(0.0, 360.0)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_atnf,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    DEC_pm,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="tab:gray",
    facecolor="tab:gray",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    DEC,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    DEC_sel,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Histrogramming the angular proper velocity components in RA and DEC.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

pm_edges = np.linspace(-200, 200, 31)

ax.hist(
    pmRA_pm,
    bins=pm_edges,
    histtype="stepfilled",
    edgecolor="tab:gray",
    facecolor="tab:gray",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    pmRA,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    pmRA_sel,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_pm,
    bins=pm_edges,
    histtype="stepfilled",
    edgecolor="tab:gray",
    facecolor="tab:gray",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    pmDEC,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    pmDEC_sel,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")

plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()